In [1]:
import torch
import torch.nn as nn
from transformers import CLIPProcessor, CLIPModel, AutoTokenizer, AutoModelForCausalLM
from PIL import Image
import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader
import json
from tqdm import tqdm
import random

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- 1. 설정 및 경로 정의 ---
print("설정을 시작합니다...")
# 모델 이름 (대회 규칙: 2024년 이전 공개, 오픈소스, 총 파라미터 3B 미만)
CLIP_MODEL_NAME = "openai/clip-vit-large-patch14" # 약 0.3B 파라미터
GPT2_MODEL_NAME = "gpt2-xl"                       # 약 1.558B 파라미터
# 총 파라미터: 0.3B + 1.558B = 1.858B. 매핑 네트워크 파라미터가 1.142B 미만이면 OK.

# VMCBench 데이터를 변환하여 저장한 경로
VMCBENCH_BASE_DIR = "VMCBench_as_Official_VQAv2_9_1" # 이 경로를 VMCBench 데이터가 있는 곳으로 변경하세요.
TRAIN_ANNOTATION_FILE = os.path.join(VMCBENCH_BASE_DIR, "vqa", "v2_OpenEnded_mscoco_train2014_annotations.json")
TRAIN_QUESTION_FILE = os.path.join(VMCBENCH_BASE_DIR, "vqa", "v2_OpenEnded_mscoco_train2014_questions.json")
TRAIN_IMAGE_DIR = os.path.join(VMCBENCH_BASE_DIR, "train2014")

# 하이퍼파라미터
BATCH_SIZE = 2 # GPU 메모리 상황에 따라 조절. GPT-2 XL은 매우 큽니다.
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4
NUM_SOFT_TOKENS = 20 # CLIP 임베딩이 변환될 가상 토큰의 개수. 10~50 사이에서 실험.

# 장치 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용할 장치: {device}")

설정을 시작합니다...
사용할 장치: cuda


In [3]:
# --- 2. 모델 및 토크나이저 로드 ---
print("모델 및 토크나이저를 로드 중입니다...")
# CLIP 모델 로드 및 가중치 고정 (학습하지 않음)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(device)
clip_model.eval() # 추론 모드
for param in clip_model.parameters():
    param.requires_grad = False
print(f"CLIP 모델 로드 완료. 파라미터 수: {sum(p.numel() for p in clip_model.parameters()) / 1e9:.3f}B")

# GPT-2 모델 로드 및 가중치 고정 (학습하지 않음)
gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL_NAME)
gpt2_model = AutoModelForCausalLM.from_pretrained(GPT2_MODEL_NAME).to(device)
gpt2_model.eval() # 추론 모드
for param in gpt2_model.parameters():
    param.requires_grad = False
# GPT-2 토크나이저에 패딩 토큰 설정 (일관성을 위해 EOS 토큰 사용)
if gpt2_tokenizer.pad_token is None:
    gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model.config.pad_token_id = gpt2_tokenizer.pad_token_id
print(f"GPT-2 모델 로드 완료. 파라미터 수: {sum(p.numel() for p in gpt2_model.parameters()) / 1e9:.3f}B")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


모델 및 토크나이저를 로드 중입니다...
CLIP 모델 로드 완료. 파라미터 수: 0.428B
GPT-2 모델 로드 완료. 파라미터 수: 1.558B


In [4]:
# --- 3. 매핑 네트워크 정의 ---
# CLIP 임베딩을 GPT-2의 소프트 프롬프트 임베딩으로 변환하는 네트워크
class ImageToSoftPromptMapping(nn.Module):
    def __init__(self, clip_embedding_dim, gpt2_hidden_dim, num_soft_tokens):
        super().__init__()
        self.num_soft_tokens = num_soft_tokens
        # 간단한 MLP 구조. 더 복잡한 구조를 실험할 수도 있습니다.
        self.mlp = nn.Sequential(
            nn.Linear(clip_embedding_dim, gpt2_hidden_dim * num_soft_tokens // 2),
            nn.ReLU(),
            nn.Linear(gpt2_hidden_dim * num_soft_tokens // 2, gpt2_hidden_dim * num_soft_tokens)
        )

    def forward(self, image_features):
        # image_features: (batch_size, clip_embedding_dim)
        mapped_features = self.mlp(image_features) # (batch_size, gpt2_hidden_dim * num_soft_tokens)
        # GPT-2 입력 임베딩과 결합하기 위해 (batch_size, num_soft_tokens, gpt2_hidden_dim) 형태로 reshape
        soft_prompt_embeddings = mapped_features.view(-1, self.num_soft_tokens, gpt2_hidden_dim)
        return soft_prompt_embeddings

CLIP_EMBEDDING_DIM = clip_model.config.projection_dim # CLIP ViT-L/14의 임베딩 차원 (768)
GPT2_HIDDEN_DIM = gpt2_model.config.hidden_size       # GPT-2 XL의 히든 스테이트 차원 (1600)

mapping_network = ImageToSoftPromptMapping(CLIP_EMBEDDING_DIM, GPT2_HIDDEN_DIM, NUM_SOFT_TOKENS).to(device)

# 매핑 네트워크의 학습 가능한 파라미터 수 확인 (총 3B 제한에 포함)
total_mapping_params = sum(p.numel() for p in mapping_network.parameters() if p.requires_grad)
print(f"매핑 네트워크 학습 가능한 파라미터: {total_mapping_params / 1e6:.2f}M")
print(f"전체 모델 총 파라미터: {(sum(p.numel() for p in clip_model.parameters()) + sum(p.numel() for p in gpt2_model.parameters()) + total_mapping_params) / 1e9:.3f}B")

매핑 네트워크 학습 가능한 파라미터: 524.34M
전체 모델 총 파라미터: 2.510B


In [5]:
# --- 4. VMCBench 데이터셋 클래스 정의 ---
class VMCBenchDataset(Dataset):
    def __init__(self, annotation_file, question_file, image_dir, tokenizer, clip_processor):
        self.image_dir = image_dir
        self.tokenizer = tokenizer
        self.clip_processor = clip_processor

        with open(annotation_file, 'r', encoding='utf-8') as f:
            self.annotations = json.load(f)["annotations"]
        with open(question_file, 'r', encoding='utf-8') as f:
            self.questions_data = {q["question_id"]: q for q in json.load(f)["questions"]}

        # 유효한 어노테이션만 필터링 (질문 ID가 존재하는 경우)
        self.annotations = [anno for anno in self.annotations if anno["question_id"] in self.questions_data]

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        anno = self.annotations[idx]
        question_id = anno["question_id"]
        image_id = anno["image_id"]
        
        question_info = self.questions_data[question_id]
        question_text = question_info["question"] # 질문 + A, B, C, D 보기가 이미 포함되어 있음

        # 이미지 로드
        image_filename = f"COCO_train2014_{str(image_id).zfill(12)}.jpg" # VQAv2 형식 파일명 사용
        image_path = os.path.join(self.image_dir, image_filename)
        image = Image.open(image_path).convert("RGB")

        # CLIP 이미지 전처리
        clip_image_inputs = self.clip_processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)

        # GPT-2 입력 프롬프트 구성: 질문 + 보기 + "Answer:"
        full_prompt = f"{question_text}\nAnswer:"
        gpt2_inputs = self.tokenizer(full_prompt, return_tensors="pt", padding=False, truncation=True, max_length=512)
        # padding=False는 collate_fn에서 일괄적으로 패딩하기 위함.

        # GPT-2의 목표 정답 토큰 ID (예: 'A', 'B', 'C', 'D'의 토큰 ID)
        # 어노테이션의 'answer_label' (0, 1, 2, 3)을 사용하여 'A', 'B', 'C', 'D' 문자열 생성
        target_char_label = chr(ord('A') + anno["answer_label"])
        target_ids = self.tokenizer(target_char_label, return_tensors="pt")["input_ids"].squeeze(0)
        
        # GPT-2의 최종 입력 시퀀스와 레이블 구성
        # 레이블은 input_ids를 오른쪽으로 한 칸 시프트한 것.
        # 단, 학습하지 않을 부분(-100 마스킹)과 정답 토큰을 정확히 지정해야 함.
        # GPT-2의 입력은 [soft_prompt_embeddings] + [input_embeddings (from gpt2_input_ids)]
        # GPT-2의 레이블은 [-100... (soft tokens)] + [-100... (prompt tokens)] + [target_ids]
        
        # GPT-2 입력 토큰 ID 시퀀스와 타겟 토큰 ID 시퀀스 결합
        # Labels will be padded in collate_fn
        combined_input_and_target_ids = torch.cat([gpt2_inputs["input_ids"].squeeze(0), target_ids], dim=0)

        return {
            "clip_image_inputs": clip_image_inputs,
            "gpt2_input_ids_for_collate": gpt2_inputs["input_ids"].squeeze(0), # Original input for GPT-2 text part
            "attention_mask_for_collate": gpt2_inputs["attention_mask"].squeeze(0),
            "labels_for_collate": combined_input_and_target_ids # This contains prompt and target answer ID
        }


In [6]:
# --- 5. Custom Collate Function ---
# DataLoader에서 배치를 처리할 때 패딩을 올바르게 수행
def collate_fn(batch):
    clip_image_inputs = torch.stack([item['clip_image_inputs'] for item in batch])
    
    # GPT-2 텍스트 입력과 전체 시퀀스(입력+타겟)의 최대 길이 계산
    max_len_text_input = max(item['gpt2_input_ids_for_collate'].size(0) for item in batch)
    max_len_full_sequence = max(item['labels_for_collate'].size(0) for item in batch)
    
    padded_gpt2_input_ids = []
    padded_attention_mask = []
    padded_labels = []

    for item in batch:
        gpt2_input_ids = item['gpt2_input_ids_for_collate']
        attention_mask = item['attention_mask_for_collate']
        full_sequence_labels = item['labels_for_collate'] # This contains input_ids + target_ids

        # Pad GPT-2 text input IDs
        pad_len_input = max_len_text_input - gpt2_input_ids.size(0)
        padded_gpt2_input_ids.append(torch.cat([gpt2_input_ids, torch.full((pad_len_input,), gpt2_tokenizer.pad_token_id, dtype=torch.long)], dim=0))
        
        # Pad attention mask for GPT-2 text input
        padded_attention_mask.append(torch.cat([attention_mask, torch.zeros(pad_len_input, dtype=torch.long)], dim=0))
        
        # Pad labels for the full sequence. Important: pad with -100 (loss ignored)
        pad_len_labels = max_len_full_sequence - full_sequence_labels.size(0)
        # Labels for the prompt part (including soft tokens) should be -100
        # Only the actual target character token should have its ID as label.
        
        # Create labels: -100 for all tokens except the actual answer token.
        # Original text part of the prompt
        labels_base = torch.full_like(gpt2_input_ids, -100) # Mask all prompt tokens
        labels_final = torch.cat([labels_base, full_sequence_labels[-1:].to(labels_base.dtype)], dim=0) # Only last token (answer char) is the target
        
        padded_labels.append(torch.cat([labels_final, torch.full((pad_len_labels,), -100, dtype=torch.long)], dim=0))


    return {
        "clip_image_inputs": clip_image_inputs,
        "gpt2_input_ids": torch.stack(padded_gpt2_input_ids), # For GPT-2's internal embeddings
        "attention_mask": torch.stack(padded_attention_mask), # For GPT-2's text part
        "labels": torch.stack(padded_labels) # Full sequence labels with -100 padding
    }

In [ ]:
# --- 6. DataLoader 생성 ---
print("데이터로더를 생성 중입니다...")
train_dataset = VMCBenchDataset(
    annotation_file=TRAIN_ANNOTATION_FILE,
    question_file=TRAIN_QUESTION_FILE,
    image_dir=TRAIN_IMAGE_DIR,
    tokenizer=gpt2_tokenizer,
    clip_processor=clip_processor
)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
print(f"훈련 데이터셋 크기: {len(train_dataset)}")
print(f"훈련 데이터로더 배치 수: {len(train_dataloader)}")

# VMCBench_as_Official_VQAv2_9_1/vqa/v2_OpenEnded_mscoco_train2014_annotations.json
# VMCBench_as_Official_VQAv2_9_1/vqa/v2_OpenEnded_mscoco_val2014_questions.json
# /data/2_data_server/cv-07/dice/2025_samsung_challenge/finetuning/VMCBench_as_Official_VQAv2_9_1/vqa/v2_mscoco_train2014_annotations.json
# /data/2_data_server/cv-07/dice/2025_samsung_challenge/finetuning/VMCBench_as_Official_VQAv2_9_1/vqa/v2_mscoco_val2014_annotations.json
# /data/2_data_server/cv-07/dice/2025_samsung_challenge/finetuning/VMCBench_as_Official_VQAv2_9_1/vqa/v2_OpenEnded_mscoco_train2014_questions.json
# /data/2_data_server/cv-07/dice/2025_samsung_challenge/finetuning/VMCBench_as_Official_VQAv2_9_1/vqa/v2_OpenEnded_mscoco_val2014_questions.json

데이터로더를 생성 중입니다...


FileNotFoundError: [Errno 2] No such file or directory: 'VMCBench_as_Official_VQAv2_9_1/vqa/v2_OpenEnded_mscoco_train2014_annotations.json'

In [9]:
# --- 7. 학습 루프 ---
print("매핑 네트워크 학습을 시작합니다...")
optimizer = torch.optim.AdamW(mapping_network.parameters(), lr=LEARNING_RATE) # 매핑 네트워크만 학습
scaler = torch.cuda.amp.GradScaler() # Mixed precision training (optional, but recommended for large models)

for epoch in range(NUM_EPOCHS):
    mapping_network.train()
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for batch in progress_bar:
        optimizer.zero_grad()

        clip_image_inputs = batch["clip_image_inputs"].to(device)
        gpt2_input_ids = batch["gpt2_input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.no_grad(): # CLIP은 고정되어 있으므로 no_grad 블록 사용
            image_features = clip_model.get_image_features(pixel_values=clip_image_inputs)
        
        # 매핑 네트워크를 통해 소프트 프롬프트 임베딩 생성
        soft_prompt_embeddings = mapping_network(image_features) # (batch_size, NUM_SOFT_TOKENS, gpt2_hidden_dim)

        # GPT-2 텍스트 입력의 임베딩 가져오기
        # GPT-2의 'wte' (word token embeddings) 모듈을 통해 입력 토큰 ID를 임베딩으로 변환
        input_embeddings = gpt2_model.get_input_embeddings()(gpt2_input_ids)
        
        # 소프트 프롬프트 임베딩과 텍스트 입력 임베딩을 결합 (소프트 프롬프트가 앞에 위치)
        extended_input_embeddings = torch.cat((soft_prompt_embeddings, input_embeddings), dim=1)
        
        # 소프트 프롬프트에 대한 어텐션 마스크 생성 (모두 1)
        soft_prompt_attention_mask = torch.ones(soft_prompt_embeddings.shape[0], soft_prompt_embeddings.shape[1], dtype=torch.long).to(device)
        # 기존 텍스트 어텐션 마스크와 결합
        extended_attention_mask = torch.cat((soft_prompt_attention_mask, attention_mask), dim=1)

        # GPT-2 레이블 조정: 소프트 프롬프트 길이만큼 레이블도 확장하고 마스킹
        # GPT-2의 Labels는 inputs_embeds의 다음 토큰에 대한 예측을 의미.
        # 따라서, extended_input_embeddings의 길이에 맞춰 labels도 확장되어야 함.
        # soft_prompt_embeddings와 input_embeddings에 해당하는 레이블은 -100으로 마스킹
        # 실제 레이블은 맨 마지막에 붙은 정답 토큰 (A, B, C, D)에만 해당되도록 설정.
        
        # Create a new labels tensor with -100 for all but the actual answer token
        # The true answer token is located at the end of the `labels` from `collate_fn`.
        # Its original position in `full_sequence_labels` was `gpt2_input_ids.size(0)`.
        # After prepending `NUM_SOFT_TOKENS`, its new position is `NUM_SOFT_TOKENS + gpt2_input_ids.size(0)`.
        
        # We need to construct the labels tensor to match `extended_input_embeddings` length
        # And ensure only the actual target token ID is present, others are -100.
        
        # The `labels` from `collate_fn` is already correctly structured and padded:
        # [-100, -100, ..., -100 (for gpt2_input_ids tokens), target_token_id, -100, ...]
        # We need to prepend NUM_SOFT_TOKENS of -100 to this.
        adjusted_labels = torch.cat([
            torch.full((labels.shape[0], NUM_SOFT_TOKENS), -100, dtype=torch.long).to(device),
            labels # The labels already have prompt masked, only the target character is relevant
        ], dim=1)
        
        # Ensure adjusted_labels match the length of extended_input_embeddings
        # This is crucial for GPT-2's forward pass.
        # If max_len_full_sequence was used in collate_fn, they should match.
        
        # Mixed precision training (optional)
        with torch.cuda.amp.autocast():
            outputs = gpt2_model(
                inputs_embeds=extended_input_embeddings,
                attention_mask=extended_attention_mask,
                labels=adjusted_labels # GPT-2 will compute loss with these labels
            )

            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=total_loss / (progress_bar.n + 1))

    print(f"Epoch {epoch+1} finished. Average Loss: {total_loss / len(train_dataloader):.4f}")


매핑 네트워크 학습을 시작합니다...


/tmp/ipykernel_3586345/1031163406.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # Mixed precision training (optional, but recommended for large models)


NameError: name 'train_dataloader' is not defined

In [ ]:

# --- 8. 학습된 매핑 네트워크 가중치 저장 ---
mapping_network_save_path = "mapping_network_weights.pth"
torch.save(mapping_network.state_dict(), mapping_network_save_path)
print(f"매핑 네트워크 학습 완료 및 가중치 저장됨: {mapping_network_save_path}")

# --- 9. 추론 예시 (학습된 모델 사용) ---
print("\n--- 추론 예시 ---")

# 모델 로드 (학습된 매핑 네트워크 가중치)
loaded_mapping_network = ImageToSoftPromptMapping(CLIP_EMBEDDING_DIM, GPT2_HIDDEN_DIM, NUM_SOFT_TOKENS).to(device)
loaded_mapping_network.load_state_dict(torch.load(mapping_network_save_path))
loaded_mapping_network.eval() # 추론 모드

def predict_answer(image_path, question, choices):
    # 이미지 로드 및 CLIP 임베딩 추출
    image = Image.open(image_path).convert("RGB")
    clip_inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        image_features = clip_model.get_image_features(pixel_values=clip_inputs["pixel_values"])
    
    # 매핑 네트워크를 통해 소프트 프롬프트 임베딩 생성
    with torch.no_grad():
        soft_prompt_embeddings = loaded_mapping_network(image_features)

    # GPT-2 프롬프트 구성 (질문 + 보기)
    choices_text = [
        f"A. {choices[0]}",
        f"B. {choices[1]}",
        f"C. {choices[2]}",
        f"D. {choices[3]}"
    ]
    full_question_text = f"{question}\n{'\n'.join(choices_text)}\nAnswer:"
    gpt2_text_inputs = gpt2_tokenizer(full_question_text, return_tensors="pt").to(device)

    # GPT-2 텍스트 입력의 임베딩 가져오기
    input_embeddings = gpt2_model.get_input_embeddings()(gpt2_text_inputs["input_ids"])

    # 소프트 프롬프트와 텍스트 임베딩 결합
    extended_input_embeddings = torch.cat((soft_prompt_embeddings, input_embeddings), dim=1)
    
    # 어텐션 마스크 확장
    soft_prompt_attention_mask = torch.ones(soft_prompt_embeddings.shape[0], soft_prompt_embeddings.shape[1], dtype=torch.long).to(device)
    extended_attention_mask = torch.cat((soft_prompt_attention_mask, gpt2_text_inputs["attention_mask"]), dim=1)

    # GPT-2 답변 생성 (다음 토큰을 예측하도록)
    with torch.no_grad():
        # GPT-2는 다음 토큰을 생성합니다. 우리는 'A', 'B', 'C', 'D' 중 하나를 기대합니다.
        # max_new_tokens=1로 설정하여 한 개의 토큰만 생성하도록 제한합니다.
        output_ids = gpt2_model.generate(
            inputs_embeds=extended_input_embeddings,
            attention_mask=extended_attention_mask,
            max_new_tokens=1, # 오직 하나의 정답 문자만 생성
            num_beams=1,      # 빔 서치 사용 안함 (더 빠름, 때로는 빔 서치보다 직접 생성 좋음)
            do_sample=False,  # 샘플링 사용 안함 (가장 높은 확률 토큰 선택)
            pad_token_id=gpt2_tokenizer.pad_token_id
        )
    
    # 생성된 토큰에서 원래 프롬프트 부분 제거 후 디코딩
    # output_ids는 [prompt_tokens, generated_token] 형태로 나옴
    # 우리가 원하는 것은 생성된 마지막 토큰
    generated_token_id = output_ids[0, extended_input_embeddings.shape[1]]
    predicted_char = gpt2_tokenizer.decode(generated_token_id, skip_special_tokens=True).strip()

    # GPT-2가 'A', 'B', 'C', 'D' 중 하나를 정확히 생성하도록 학습하는 것이 중요합니다.
    # 만약 엉뚱한 토큰이 생성된다면, 프롬프트 엔지니어링이나 학습 데이터의 다양성을 고려해야 합니다.
    
    # 결과가 'A', 'B', 'C', 'D' 중 하나가 아닐 경우 처리 (Fallback)
    if predicted_char not in ['A', 'B', 'C', 'D']:
        print(f"경고: 예상치 못한 답변 생성: '{predicted_char}'. 기본값 'A'로 설정합니다.")
        return 'A' # Fallback to 'A' if invalid char is generated

    return predicted_char

# --- 테스트 데이터셋으로 추론 예시 ---
# 대회에서 제공된 test dataset 경로를 사용합니다.
TEST_CSV_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv"
TEST_IMAGE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images"

df_test_input = pd.read_csv(TEST_CSV_PATH)
df_submission = pd.DataFrame(columns=["ID", "answer"])

print("\n대회 테스트 데이터셋으로 추론 시작...")
for index, row in tqdm(df_test_input.iterrows(), total=len(df_test_input), desc="Generating predictions"):
    img_id = row['ID']
    img_path = os.path.join(TEST_IMAGE_DIR, os.path.basename(row['img_path']))
    question = row['Question']
    choices = [row['A'], row['B'], row['C'], row['D']]

    predicted_answer = predict_answer(img_path, question, choices)
    
    df_submission = pd.concat([df_submission, pd.DataFrame([{"ID": img_id, "answer": predicted_answer}])], ignore_index=True)

# 최종 제출 파일 저장
SUBMISSION_FILE_PATH = "submission.csv"
df_submission.to_csv(SUBMISSION_FILE_PATH, index=False)
print(f"\n최종 제출 파일 저장 완료: {SUBMISSION_FILE_PATH}")